## importing

In [ ]:
import pandas as pd
import numpy as np
from time import time
from tqdm import tqdm
import pickle

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, recall_score, precision_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf
from tensorflow.keras.models import load_model

from functions.data_preparation import create_encoded_categories, mean_encoding

import seaborn as sns
import matplotlib.pyplot as plt

## data preparation

In [2]:
kliki_do_zgloszenia = pd.read_csv('data/kliki_do_zgloszenia.csv', sep = ';')
kliki_uczenie = pd.read_csv('data/kliki_uczenie.csv', sep = ';')
zgloszenie = pd.read_csv('data/zgloszenie_DS_Nazwisko_Imie_RRRRMMDD.csv', sep = ';')

In [123]:
kliki_uczenie.iloc[ : 5, : 10]

,id,klik,data_godzina,baner_pozycja,strona_id,strona_domena,strona_kategoria,aplikacja_id,aplikacja_domena,aplikacja_kategoria
0,93718246913880603,0,17030100,0,85f751fd,c4e18dd6,50e219e0,febd1138,82e27996,0f2161f8
1,93718246913880604,0,17030100,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22
2,93718246913880605,0,17030100,0,543a539e,c7ca3108,3e814130,ecad2386,7801e8d9,07d7df22
3,93718246913880606,0,17030100,1,e151e245,7e091613,f028772b,ecad2386,7801e8d9,07d7df22
4,93718246913880607,0,17030100,0,85f751fd,c4e18dd6,50e219e0,39947756,2347f47a,cef3e649


In [124]:
kliki_uczenie.iloc[ : 5, 10 : ]

,urz_id,urz_ip,urz_model,urz_typ,urz_polaczenie,kat1,kat2,kat3,kat4,kat5,kat6,kat7,kat8,kat9
0,a99f214a,98fc2769,3eb8d368,1,0,105,119743,10320,10050,12264,13,10427,100000,61
1,a99f214a,7357d5f1,ecb851b2,1,0,105,115706,10320,10050,11722,10,10035,-1,79
2,a99f214a,897cea85,1974edc5,1,0,105,120352,10320,10050,12333,10,10039,-1,157
3,a99f214a,9758e3a7,b3f4c8ce,1,0,105,117037,10320,10050,11934,12,10039,-1,16
4,405cbd4b,fa0ded7b,a9e0c3ab,1,2,105,118993,10320,10050,12161,10,10035,100148,157


In [10]:
kliki_uczenie.shape

(3619621, 24)

In [ ]:
columns_to_encode = ['strona_id', 'strona_domena', 'strona_kategoria', 'aplikacja_id', 'aplikacja_domena', 'aplikacja_kategoria', 'urz_id', 'urz_ip', 'urz_model']
data = kliki_uczenie
# data = kliki_uczenie.iloc[ : len(kliki_uczenie) // 2]
create_encoded_categories(data, columns_to_encode)

100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:07<00:00,  1.27it/s]


In [ ]:
columns_to_encode = ['strona_id', 'strona_domena', 'strona_kategoria', 'aplikacja_id', 'aplikacja_domena', 'aplikacja_kategoria', 'urz_id', 'urz_ip', 'urz_model']
encoded_categories = pd.read_csv('data/encoded_categories.csv')
kliki_uczenie_encoded = mean_encoding(kliki_uczenie, columns_to_encode, encoded_categories)

100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:27<00:00,  3.06s/it]


In [280]:
kliki_uczenie_encoded.iloc[ : 5, : 15]

,id,klik,data_godzina,baner_pozycja,urz_typ,urz_polaczenie,kat1,kat2,kat3,kat4,kat5,kat6,kat7,kat8,kat9
0,93718246913894435,0,17030101,0,1,0,105,120366,10320,10050,12333,10,10039,-1,157
1,93718246917082238,0,17030821,0,1,0,105,118949,10216,10036,12153,13,10427,100062,61
2,93718246917377725,0,17030914,0,1,0,105,118945,10320,10050,12153,13,10427,100062,61
3,93718246914330815,0,17030203,0,1,0,105,112026,10320,10050,11248,12,10039,100205,13
4,93718246914955793,0,17030307,1,1,0,105,120363,10216,10036,12333,10,10039,-1,157


In [281]:
kliki_uczenie_encoded.iloc[ : 5, 15 : ]

,strona_id_encoded,strona_domena_encoded,strona_kategoria_encoded,aplikacja_id_encoded,aplikacja_domena_encoded,aplikacja_kategoria_encoded,urz_id_encoded,urz_ip_encoded,urz_model_encoded
0,0.209982,0.209982,0.178911,0.197836,0.194042,0.198436,0.173876,0.0,0.214912
1,0.206307,0.206307,0.208187,0.197836,0.194042,0.198436,0.173876,0.0,0.214912
2,0.019943,0.019943,0.040092,0.197836,0.194042,0.198436,0.173876,0.0,0.214912
3,0.117175,0.120965,0.127216,0.193199,0.188076,0.096510,0.000000,0.5,0.214912
4,0.296691,0.255543,0.178911,0.197836,0.194042,0.198436,0.173876,0.5,0.214912


## checking outliers

potential outliers to remove  
urz_typ = 2  
kat5, kat7, kat8, strona_kategoria, aplikacja_domena, aplikacja_kategoria- delete values which occurs less than 50 times  
strona_id, strona_domena, aplikacja_id, urz_id, urz_ip, urz_model, kat2- delete whole column

In [478]:
df[df['no_occurences'] < 50]

,strona_domena,no_occurences
0,75756269,1
1,44f3f8e2,1
2,e2e347b3,1
3,fb1ae82b,1
4,9e13de5d,1
...,...,...
3318,c46b4d51,48
3319,d6fd405a,48
3320,3d3f385d,48
3321,e3491659,48


In [476]:
i = 5
# i -= 1
column = kliki_uczenie.columns[i]
y = kliki_uczenie[column].values
i += 1

obj = np.unique(y, return_counts = True)
df = pd.DataFrame([obj[0], obj[1]]).T
df = df.sort_values(by = 1).reset_index(drop = True)
df.columns = [column, 'no_occurences']
df
# df[df[1] < 50]
# # plt.plot(df[1].iloc[ : 2000])

,strona_domena,no_occurences
0,75756269,1
1,44f3f8e2,1
2,e2e347b3,1
3,fb1ae82b,1
4,9e13de5d,1
...,...,...
4147,98572c79,92469
4148,7687a86e,115167
4149,7e091613,302658
4150,f3845767,590538


## model testing

In [4]:
models_performance = pd.DataFrame()

In [ ]:
# trying different parameters for logistic regression
# penalty l1 and elasticnet doesnt work somehow

x = kliki_uczenie_encoded.drop(['klik', 'id', 'data_godzina'], axis = 1).values
y = kliki_uczenie_encoded.klik.values
# standardize x
x = StandardScaler().fit_transform(x)

# x = x.iloc[ : 200]
# x = x[ : 200]
# y = y.iloc[ : 200]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.25)

# models_parameters = {'C': [0.01, 0.1, 1, 10, 100], 'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'], 'penalty': ['none', 'l1', 'l2', 'elasticnet']}
models_parameters = {'C': [0.01, 1, 100], 'solver': ['saga', 'newton-cg', 'liblinear'], 'penalty': ['none', 'elasticnet']}

model = GridSearchCV(estimator = LogisticRegression(max_iter = 1000), param_grid = models_parameters)
model_name = str(model.estimator).split('(')[0]
model.fit(x_train, y_train)

# save model
model.save(f'models/{model_name}')

# save info about model performance in table models_performance
df = pd.DataFrame([[model_name, test_score, params] for test_score, params in zip(model.cv_results_['mean_test_score'], model.cv_results_['params'])])
df.columns = ['model_name', 'score', 'model_parameters']

models_performance = pd.concat((models_performance, df), axis = 1)

c:\python\python37\lib\site-packages\sklearn\linear_model\_logistic.py:1321: UserWarning: Setting penalty='none' will ignore the C and l1_ratio parameters
  "Setting penalty='none' will ignore the C and l1_ratio "
c:\python\python37\lib\site-packages\sklearn\linear_model\_logistic.py:1321: UserWarning: Setting penalty='none' will ignore the C and l1_ratio parameters
  "Setting penalty='none' will ignore the C and l1_ratio "
c:\python\python37\lib\site-packages\sklearn\linear_model\_logistic.py:1321: UserWarning: Setting penalty='none' will ignore the C and l1_ratio parameters
  "Setting penalty='none' will ignore the C and l1_ratio "
c:\python\python37\lib\site-packages\sklearn\linear_model\_logistic.py:1321: UserWarning: Setting penalty='none' will ignore the C and l1_ratio parameters
  "Setting penalty='none' will ignore the C and l1_ratio "
c:\python\python37\lib\site-packages\sklearn\linear_model\_logistic.py:1321: UserWarning: Setting penalty='none' will ignore the C and l1_ratio 

In [ ]:
# trying different parameters for knn

x = kliki_uczenie_encoded.drop(['klik', 'id', 'data_godzina'], axis = 1).values
y = kliki_uczenie_encoded.klik.values

# x = x.iloc[ : 200]
# x = x[ : 200]
# y = y.iloc[ : 200]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.25)

# models_parameters = {'n_neighbors': [i for i in range(1, 10)], 'metric': ['euclidean', 'manhattan', 'chebyshev', 'minkowski', 'wminkowski', 'seuclidean', 'mahalanobis']}
models_parameters = {'n_neighbors': [np.sqrt(len(x_train)), np.sqrt(len(x_train)) / 10, np.sqrt(len(x_train)) / 40], 'metric': ['euclidean']}

model = GridSearchCV(estimator = KNeighborsClassifier(), param_grid = models_parameters)
model_name = str(model.estimator).split('(')[0]
model.fit(x_train, y_train)

# save model
model.save(f'models/{model_name}')

# save info about model performance in table models_performance
df = pd.DataFrame([[model_name, test_score, params] for test_score, params in zip(model.cv_results_['mean_test_score'], model.cv_results_['params'])])
df.columns = ['model_name', 'score', 'model_parameters']

models_performance = pd.concat((models_performance, df))

In [ ]:
# trying different parameters for RandomForestClassifier

x = kliki_uczenie_encoded.drop(['klik', 'id', 'data_godzina'], axis = 1).values
y = kliki_uczenie_encoded.klik.values

# x = x.iloc[ : 200]
# x = x[ : 200]
# y = y.iloc[ : 200]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.25)

models_parameters = {'n_estimators ': [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)], 'max_features': ['sqrt', 'log2'], 
                     'max_depth': [int(x) for x in np.linspace(10, 110, num = 11)], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4], 
                     'bootstrap': [True, False]}

model = GridSearchCV(estimator = RandomForestClassifier(), param_grid = models_parameters)
model_name = str(model.estimator).split('(')[0]
model.fit(x_train, y_train)

# save model
model.save(f'models/{model_name}')

# save info about model performance in table models_performance
df = pd.DataFrame([[model_name, test_score, params] for test_score, params in zip(model.cv_results_['mean_test_score'], model.cv_results_['params'])])
df.columns = ['model_name', 'score', 'model_parameters']

models_performance = pd.concat((models_performance, df))

In [ ]:
# # trying different parameters for DecisionTreeClassifier

# models_parameters = {'criterion': ['gini', 'entropy'], 'splitter': ['best', 'random'], 'min_samples_split': [2, 5, 10], 
#                      'min_samples_leaf': [1, 2, 4], 'max_features': ['none', 'sqrt', 'log2'], 'random_state': [0, 1, 10, 20, 42], 
#                      'min_impurity_decrease': [0, 0.1, 0.5, 1, 10, 50], 'max_depth': [int(x) for x in np.linspace(10, 110, num = 11)]}

# model = GridSearchCV(estimator = DecisionTreeClassifier(), param_grid = models_parameters)
# model_name = str(model.estimator).split('(')[0]
# model.fit(x_train, y_train)

# # save model
# pickle.dump(model, open(f'models/{model_name}', 'wb'))

# # save info about model performance in table models_performance
# df = pd.DataFrame([[model_name, test_score, params] for test_score, params in zip(model.cv_results_['mean_test_score'], model.cv_results_['params'])])
# df.columns = ['model_name', 'score', 'model_parameters']

# models_performance = pd.concat((models_performance, df))

potential outliers to remove:  
urz_typ = 2  
kat5, kat7, kat8, strona_kategoria, aplikacja_domena, aplikacja_kategoria- delete values which occurs less than 50 times  
strona_id, strona_domena, aplikacja_id, urz_id, urz_ip, urz_model, kat2- delete whole column

In [ ]:
# # removing outliers from x

# data = kliki_uczenie
# print('length of data before removing outliers: ', len(data))

# for column in tqdm(['urz_typ', 'kat5', 'kat7', 'kat8', 'strona_kategoria', 'aplikacja_domena', 'aplikacja_kategoria']):
#     df = data.groupby(column).count().iloc[:, 0].reset_index()
#     df.columns = [column, 'no_occurences']
#     df = df.sort_values('no_occurences')
#     values_to_remove = df[df.no_occurences < 50][column].values
    
#     data = data[data[column].apply(lambda x: x not in values_to_remove)]
    
# print('length of data after removing outliers: ', len(data))

# # encoding
# columns_to_encode = ['strona_id', 'strona_domena', 'strona_kategoria', 'aplikacja_id', 'aplikacja_domena', 'aplikacja_kategoria', 'urz_id', 'urz_ip', 'urz_model']
# encoded_categories = pd.read_csv('data/encoded_categories.csv')
# data = mean_encoding(data, columns_to_encode, encoded_categories)

# y = data.klik.values
# x = data.drop(['klik', 'id', 'data_godzina'], axis = 1).values

length of data before removing outliers:  3619621


100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [01:48<00:00, 15.47s/it]


length of data after removing outliers:  3617138


100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:28<00:00,  3.20s/it]


In [466]:
# removing unnecessary columns
# x = data.drop(['klik', 'id', 'data_godzina', 'strona_id_encoded', 'strona_domena_encoded', 'aplikacja_id_encoded',
#                'urz_id_encoded', 'urz_ip_encoded', 'urz_model_encoded', 'kat2'], axis = 1).values
# x = data.drop(['klik', 'id', 'data_godzina', 'strona_id_encoded', 'aplikacja_id_encoded',
#                'kat2'], axis = 1).values

In [467]:
# neural network

x = kliki_uczenie_encoded.drop(['klik', 'id', 'data_godzina'], axis = 1)
y = kliki_uczenie_encoded.klik.values

# normalizing data
mean = np.mean(x, axis = 0)
std = np.std(x, axis = 0)
x -= mean
x /= std

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.25)
# prepare categorical format of y for training
y_train_cat = tf.keras.utils.to_categorical(y_train)

model = Sequential()
model.add(Dense(64, input_shape = (x_train.shape[1], ), activation = 'sigmoid'))
model.add(Dropout(0.2))
model.add(Dense(32, activation = 'sigmoid'))
model.add(Dropout(0.2))
model.add(Dense(2, activation = 'softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

history = model.fit(x = x_train,
                    y = y_train_cat,
                    batch_size = 128,
                    epochs = 1,
                    validation_split = 0.2,
                    callbacks = [tf.keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = 5)],
                    verbose = 1)

model._normalization_mean = tf.Variable([mean], trainable = False)
model._normalization_std  = tf.Variable([std], trainable = False)

# model.save('models/nn_model')

Model: "sequential_38"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_114 (Dense)            (None, 64)                1216      
_________________________________________________________________
dropout_83 (Dropout)         (None, 64)                0         
_________________________________________________________________
dense_115 (Dense)            (None, 32)                2080      
_________________________________________________________________
dropout_84 (Dropout)         (None, 32)                0         
_________________________________________________________________
dense_116 (Dense)            (None, 2)                 66        
Total params: 3,362
Trainable params: 3,362
Non-trainable params: 0
_________________________________________________________________
None
16956/16956 [==============================] - 19s 1ms/step - loss: 0.2120 - accuracy: 0.9076 - val_loss: 0.2004

In [284]:
model = load_model('models/nn_model')

In [283]:
model._normalization_std.numpy()

array([[5.10869775e-01, 5.41444653e-01, 8.56941046e-01, 1.11721669e+00,
        4.98134246e+03, 2.15588528e+01, 4.76175505e+01, 6.10286124e+02,
        1.32312559e+00, 3.49955302e+02, 4.99308414e+04, 6.70069954e+01,
        1.04709992e-01, 9.85661944e-02, 4.76156429e-02, 7.25013812e-02,
        5.13594959e-02, 4.38066815e-02, 1.38455077e-01, 2.60029464e-01,
        6.75697981e-02]])

In [285]:
pred = model.predict(x_test)
pred = np.argmax(pred, axis = 1)
confusion_matrix(y_test, pred)

array([[744480,   7157],
       [ 74832,  78437]], dtype=int64)

In [286]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.91      0.99      0.95    751637
           1       0.92      0.51      0.66    153269

    accuracy                           0.91    904906
   macro avg       0.91      0.75      0.80    904906
weighted avg       0.91      0.91      0.90    904906



## task

In [130]:
kliki_do_zgloszenia.head()

,id,data_godzina,baner_pozycja,strona_id,strona_domena,strona_kategoria,aplikacja_id,aplikacja_domena,aplikacja_kategoria,urz_id,...,urz_polaczenie,kat1,kat2,kat3,kat4,kat5,kat6,kat7,kat8,kat9
0,93718246917500224,17031000,1,d9750ee7,98572c79,f028772b,ecad2386,7801e8d9,07d7df22,a99f214a,...,0,105,117614,10320,10050,11993,12,11063,100083,33
1,93718246917500225,17031000,0,2ac17d2f,e8f9fead,3e814130,ecad2386,7801e8d9,07d7df22,a99f214a,...,0,105,117239,10320,10050,11973,13,10039,100103,23
2,93718246917500226,17031000,0,9a977531,a434fa42,f028772b,ecad2386,7801e8d9,07d7df22,a99f214a,...,0,105,120009,10320,10050,12283,10,10163,-1,95
3,93718246917500227,17031000,1,e151e245,7e091613,f028772b,ecad2386,7801e8d9,07d7df22,a99f214a,...,0,105,123723,10320,10050,12716,13,10047,-1,23
4,93718246917500228,17031000,0,85f751fd,c4e18dd6,50e219e0,7358e05e,b9528b13,cef3e649,ba7781a5,...,0,105,122624,10320,10050,12374,13,10039,-1,23


In [132]:
zgloszenie.head()

,id,klik
0,93718246917500224,0
1,93718246917500225,0
2,93718246917500226,0
3,93718246917500227,0
4,93718246917500228,0


In [288]:
columns_to_encode = ['strona_id', 'strona_domena', 'strona_kategoria', 'aplikacja_id', 'aplikacja_domena', 'aplikacja_kategoria', 'urz_id', 'urz_ip', 'urz_model']
kliki_do_zgloszenia_encoded = mean_encoding(kliki_do_zgloszenia, columns_to_encode, encoded_categories)

100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:07<00:00,  1.22it/s]


In [289]:
kliki_do_zgloszenia_encoded.iloc[ : 5, : 15]

,id,data_godzina,baner_pozycja,urz_typ,urz_polaczenie,kat1,kat2,kat3,kat4,kat5,kat6,kat7,kat8,kat9,strona_id_encoded
0,93718246917560672,17031004,1,1,0,105,118949,10216,10036,12153,13,10427,100062,61,0.296691
1,93718246917595934,17031005,0,1,0,105,123804,10320,10050,12726,13,10803,-1,229,0.117175
2,93718246917628901,17031007,0,1,0,105,119016,10300,10250,12162,12,10039,-1,33,0.470469
3,93718246917559798,17031004,0,1,0,105,119016,10300,10250,12162,12,10039,-1,33,0.466845
4,93718246917635614,17031008,0,1,0,105,106563,10320,10050,10572,12,10039,-1,32,0.171062


In [290]:
kliki_do_zgloszenia_encoded.iloc[ : 5, 15 : ]

,strona_domena_encoded,strona_kategoria_encoded,aplikacja_id_encoded,aplikacja_domena_encoded,aplikacja_kategoria_encoded,urz_id_encoded,urz_ip_encoded,urz_model_encoded
0,0.255543,0.178911,0.197836,0.194042,0.198436,0.173876,0.000000,0.214912
1,0.120965,0.127216,0.199218,0.199194,0.109450,0.000000,0.000000,0.214912
2,0.459541,0.281525,0.197836,0.194042,0.198436,0.173876,0.500000,0.214912
3,0.459541,0.281525,0.197836,0.194042,0.198436,0.173876,0.000000,0.214912
4,0.171056,0.178911,0.197836,0.194042,0.198436,0.173876,0.108108,0.214912


In [ ]:
# making predictions

model = load_model('models/nn_model')
std = model._normalization_std.numpy()
mean = model._normalization_mean.numpy()

x = kliki_do_zgloszenia_encoded.drop(['id', 'data_godzina'], axis = 1).values

# standardizing data
x -= mean
x /= std

pred = model.predict(x)

In [469]:
pred = np.argmax(pred, axis = 1)
pred

array([0, 0, 1, ..., 0, 0, 0], dtype=int64)

In [470]:
zgloszenie = pd.concat((kliki_do_zgloszenia_encoded['id'], pd.DataFrame(pred)), axis = 1)
zgloszenie.columns = ['id', 'klik']
zgloszenie.to_csv('data/zgloszenie_DS_Bulka_Marcin_20220529.csv')
zgloszenie

,id,klik
0,93718246917560672,0
1,93718246917595934,0
2,93718246917628901,1
3,93718246917559798,0
4,93718246917635614,0
...,...,...
423271,93718246917525418,0
423272,93718246917869598,0
423273,93718246917797474,0
423274,93718246917753102,0
